# Solar Filament V16 — 12-hour NEW MODEL SEARCH LAB

**Clean-slate training, not V7/V13/V14/V15 postprocessing.** Train several semantic architectures from scratch or ImageNet initialization, train YOLO11s/m instance segmenters, select individual and hybrid systems by acquisition-day held-out PQ, save checkpoints and generate a **new-model** submission.

**What is deliberately *not* required:** your old 0.34 CSV, YOLO11n model, V7 recipe or recovery hashes. The historical submission is not used to select models. The only required input is the official competition dataset with its *competition-provided* train annotations. Never attach public MAGFiLO ground-truth annotations for any competition test images.

**Kaggle setup:** GPU T4 (single T4 works; T4×2 available but this notebook uses GPU 0 to avoid unstable notebook distributed training); Internet ON if you want ImageNet pretrained weights and official YOLO weights; ~12-hour session, at least 14 GB working disk. On Internet OFF, models fall back to initialization from scratch and the notebook labels their initialization honestly. The run may not finish every planned model; checkpoints are saved after each epoch. Attach your previous V16 output as an input dataset to resume after a session reset.

**Targets:** independent validation and physically meaningful filament instance masks; *not* an inflated public ranking. The official scoring implementation remains authoritative: https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook. The code below uses an explicitly identified local approximate PQ scorer; it does not claim official parity.

In [ ]:
# 0 — USER CONTROLS. You can leave these as supplied for the first run.
from pathlib import Path
MODE = "FULL"                    # FULL / SEMANTIC_ONLY / YOLO_ONLY / SCORE_ONLY / INFER_ONLY
DATA_ROOT = Path("/kaggle/input/datasets/lighittha/solar3k/MAGFiLO_1.0_Kaggle_2026")
OUT = Path("/kaggle/working/solar_v16_new_models")
PREVIOUS_V16_ROOT = None          # e.g. /kaggle/input/datasets/lighittha/solar-v16-part1/solar_v16_new_models
SEED = 2026
MAX_WALL_HOURS = 11.4            # 12-hour session reserve
RESERVE_FINAL_HOURS = 1.5        # save enough time for ranking/export
RUN_SEMANTIC = True
RUN_YOLO = True
RUN_YOLO_M = True
TRY_IMAGENET_PRETRAINED = True   # requires Internet or pretrained timm cache
TRY_OFFICIAL_YOLO_PRETRAINED = True
SEMANTIC_CROP = 512
SEMANTIC_BATCH = 4
NUM_WORKERS = 2
USE_TTA = False                  # False for screening speed; optional True for separate later run
TOP_MODELS_FOR_TEST = 2
VALIDATION_TUNE_LIMIT = 97       # 97 = entire original tune split
YOLO_S_HOURS = 2.0
YOLO_M_HOURS = 1.15
SEMANTIC_RUNS = [
    dict(name="resunet32", kind="resunet", epochs=18, steps=180, stage_hours=1.45),
    dict(name="resnet34_fpn", kind="timm", backbone="resnet34", epochs=18, steps=180, stage_hours=1.45),
    dict(name="efficientnet_b0_fpn", kind="timm", backbone="efficientnet_b0", epochs=17, steps=180, stage_hours=1.35),
    dict(name="convnext_tiny_fpn", kind="timm", backbone="convnext_tiny", epochs=14, steps=125, stage_hours=1.10, batch=2),
]
assert MODE in ("FULL","SEMANTIC_ONLY","YOLO_ONLY","SCORE_ONLY","INFER_ONLY")
assert 0 < MAX_WALL_HOURS <= 11.8
for d in (OUT,OUT/"labels",OUT/"models",OUT/"probabilities",OUT/"reports",OUT/"yolo_data",OUT/"submissions"):
    d.mkdir(parents=True,exist_ok=True)
print("V16 clean-slate model search. Mode:",MODE,"Output:",OUT)


In [ ]:
# 1 — Dependencies, deterministic environment, run clock, resumability
import os, sys, time, json, math, random, hashlib, shutil, subprocess, importlib.util, platform, warnings, gc, traceback
from collections import defaultdict, Counter
from contextlib import nullcontext
import numpy as np, pandas as pd, cv2
from tqdm.auto import tqdm
from scipy.optimize import linear_sum_assignment

def ensure_module(module, package=None):
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",package or module])

for name,pkg in [("pycocotools","pycocotools"),("timm","timm"),("ultralytics","ultralytics")]:
    ensure_module(name,pkg)
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset,DataLoader
from pycocotools import mask as coco
from IPython.display import display
import timm

cv2.setNumThreads(0)
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda" and MODE not in ("SCORE_ONLY","INFER_ONLY"):
    raise RuntimeError("Enable a Kaggle T4 GPU for 12-hour training.")
torch.set_num_threads(min(4,os.cpu_count() or 2))
START=time.monotonic()
DEADLINE=START + MAX_WALL_HOURS*3600

def hours_left(): return (DEADLINE-time.monotonic())/3600

def have_time(hours=0.0): return hours_left() >= hours+RESERVE_FINAL_HOURS

def save_json(name,obj):
    p=OUT/"reports"/name; tmp=p.with_suffix(".tmp")
    tmp.write_text(json.dumps(obj,indent=2,default=str));tmp.replace(p)

def file_hash(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(1<<20),b""): h.update(b)
    return h.hexdigest()

INFO={"python":sys.version,"torch":torch.__version__,"timm":timm.__version__,
      "device":str(DEVICE),"gpu":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
      "mode":MODE,"seed":SEED,"max_wall_hours":MAX_WALL_HOURS}
save_json("environment.json",INFO)
print("Environment:",INFO)


In [ ]:
# 2 — Competition data only; preserve historical train/tune/audit group split if present.
ANNOT_NAME="MAGFiLO_1.0_Annotations_kaggle2026_train.json"
if not (DATA_ROOT/"train"/ANNOT_NAME).is_file():
    candidates=list(Path("/kaggle/input").rglob(ANNOT_NAME))
    if len(candidates)!=1: raise FileNotFoundError(f"Expected one official competition annotation file; found {candidates}")
    DATA_ROOT=candidates[0].parent.parent
ANNOT=DATA_ROOT/"train"/ANNOT_NAME
TRAIN_DIR=DATA_ROOT/"train"/"train_images"
TEST_DIR=DATA_ROOT/"test"/"test_images"
assert ANNOT.exists() and TRAIN_DIR.is_dir() and TEST_DIR.is_dir()
TRAIN={p.stem:p for p in TRAIN_DIR.iterdir() if p.suffix.lower() in (".jpg",".jpeg",".png")}
TEST={p.stem:p for p in TEST_DIR.iterdir() if p.suffix.lower() in (".jpg",".jpeg",".png")}
assert len(TRAIN)==707 and len(TEST)==180,(len(TRAIN),len(TEST))
DOC=json.loads(ANNOT.read_text())
ANNOT_BY_ID=defaultdict(list)
for a in DOC["annotations"]: ANNOT_BY_ID[str(a["image_id"])].append(a)
VERSION_BY_STEM=defaultdict(list)
for im in DOC["images"]:
    stem=Path(im["file_name"]).stem
    if stem in TRAIN:
        VERSION_BY_STEM[stem].append((str(im["id"]),ANNOT_BY_ID[str(im["id"])]))
assert set(VERSION_BY_STEM)==set(TRAIN)

# Reuse the known split to compare candidates fairly; no V7 model or CSV is used.
SPLIT_FILE=None
for p in Path("/kaggle/input").rglob("split_manifest.json"):
    try:
        if file_hash(p)=="f9c906951de50e5245c7411ceba5ebd093eff2368f9719c628dc8d1d7e52361c":
            SPLIT_FILE=p;break
    except Exception:pass
if SPLIT_FILE:
    SPLIT=json.loads(SPLIT_FILE.read_text())
else:
    # If the original split is unavailable, group observations by acquisition day.
    days=defaultdict(list)
    for s in TRAIN:days[s[:8]].append(s)
    keys=sorted(days);random.Random(SEED).shuffle(keys)
    bins={"train":[],"tune":[],"audit":[]}
    for day in keys:
        k=min(bins,key=lambda n:(len(bins[n])/({"train":.725,"tune":.1375,"audit":.1375}[n]),
                                {"train":0,"tune":1,"audit":2}[n]))
        bins[k].extend(days[day])
    SPLIT={k:sorted(v) for k,v in bins.items()}
    print("Original split unavailable; created fresh acquisition-day split:", {k:len(v) for k,v in SPLIT.items()})
assert set().union(*(set(v) for v in SPLIT.values()))==set(TRAIN)
assert all(not(set(SPLIT[a])&set(SPLIT[b])) for a in SPLIT for b in SPLIT if a!=b)
assert all(not({s[:8] for s in SPLIT[a]}&{s[:8] for s in SPLIT[b]}) for a in SPLIT for b in SPLIT if a!=b)
if SPLIT_FILE: assert [len(SPLIT[x]) for x in ("train","tune","audit")]==[513,97,97]
SAVE_SPLIT={k:sorted(v) for k,v in SPLIT.items()}
save_json("split_manifest.json",SAVE_SPLIT)
save_json("data_provenance.json",{"official_train_annotation_sha256":file_hash(ANNOT),
          "train_images":len(TRAIN),"test_images":len(TEST),"label_source":"competition train only",
          "split_source":str(SPLIT_FILE) if SPLIT_FILE else "created acquisition-day split"})
print("Train/tune/audit:",{k:len(v) for k,v in SPLIT.items()},"test:",len(TEST))


In [ ]:
# 3 — True COCO polygon decoding; immutable local validation instances and soft multi-annotator targets
H=W=2048

def enc(m): return coco.encode(np.asfortranarray(np.asarray(m,np.uint8)))
def asc(r):return r["counts"].decode("ascii") if isinstance(r["counts"],bytes) else r["counts"]
def as_rle(x):
    if isinstance(x,str):return {"size":[H,W],"counts":x.encode("ascii")}
    if isinstance(x,dict):
        if isinstance(x.get("counts"),list):return coco.frPyObjects(x,H,W)
        return {"size":[H,W],"counts":x["counts"].encode("ascii") if isinstance(x["counts"],str) else x["counts"]}
    raise TypeError(type(x))

def annotation_rle(a):
    s=a.get("segmentation")
    if not s:return None
    if isinstance(s,dict):return as_rle(s)
    if isinstance(s,list):
        if s and isinstance(s[0],(float,int)): s=[s]
        polys=[]
        for p in s:
            if len(p)>=6 and len(p)%2==0: polys.append(p)
        if not polys:return None
        raw=coco.frPyObjects(polys,H,W)
        return coco.merge(raw) if isinstance(raw,list) else raw
    return None

GT={}
for s in SPLIT["tune"]+SPLIT["audit"]:
    GT[s]=[[r for a in annotations if (r:=annotation_rle(a)) is not None]
           for _,annotations in VERSION_BY_STEM[s]]

# Build one 8-bit soft foreground mask / image; center coordinates for positive crop mining.
LABEL_CENTER={}
LABEL_ROOT=OUT/"labels"
LABEL_CACHE=OUT/"reports"/"label_centers.json"
if LABEL_CACHE.exists():
    LABEL_CENTER=json.loads(LABEL_CACHE.read_text())
if MODE not in ("SCORE_ONLY","INFER_ONLY"):
    for s in tqdm(sorted(set(SPLIT["train"]+SPLIT["tune"])),desc="Soft labels from authorized train annotations"):
        p=LABEL_ROOT/(s+".png")
        if p.is_file() and s in LABEL_CENTER:continue
        versions=VERSION_BY_STEM[s]
        soft=np.zeros((H,W),np.float32); centers=[]
        for _,annotations in versions:
            union=np.zeros((H,W),np.uint8)
            for a in annotations:
                rr=annotation_rle(a)
                if rr is None:continue
                area=int(coco.area(rr))
                if area<12:continue
                bb=np.asarray(coco.toBbox(rr),float)
                centers.append((int(bb[0]+bb[2]/2),int(bb[1]+bb[3]/2)))
                union|=coco.decode(rr).astype(np.uint8)
            soft+=union
        soft/=max(1,len(versions))
        ok=cv2.imwrite(str(p),np.clip(np.round(soft*255),0,255).astype(np.uint8),[cv2.IMWRITE_PNG_COMPRESSION,3])
        if not ok:raise IOError(p)
        LABEL_CENTER[s]=centers
        del soft
    save_json("label_centers.json",LABEL_CENTER)
print("Prepared label maps:",len(list(LABEL_ROOT.glob("*.png"))))


In [ ]:
# 4 — Patch generator: native 512px details, positive crop mining, CLAHE+black-hat input, augmentation
CLAHE=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
BH_KERNEL=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(19,19))

def make_channels(gray):
    enhanced=CLAHE.apply(gray)
    dark=cv2.morphologyEx(enhanced,cv2.MORPH_BLACKHAT,BH_KERNEL)
    return np.stack([gray,enhanced,dark],-1)

def load_patches(stem,idx,train_mode,epoch,crop=512):
    rng=np.random.default_rng(SEED+int(idx)*2377+int(epoch)*1000003)
    gray=cv2.imread(str(TRAIN[stem]),cv2.IMREAD_GRAYSCALE)
    lab=cv2.imread(str(LABEL_ROOT/(stem+".png")),cv2.IMREAD_GRAYSCALE)
    if gray is None or lab is None or gray.shape!=(2048,2048) or lab.shape!=(2048,2048):raise RuntimeError(stem)
    centers=LABEL_CENTER.get(stem,[])
    if centers and rng.random()<(.82 if train_mode else .60):
        cx,cy=centers[int(rng.integers(len(centers)))]
        jitter=int(crop*.29)
        cx+=int(rng.integers(-jitter,jitter+1));cy+=int(rng.integers(-jitter,jitter+1))
    else:cx,cy=int(rng.integers(0,W)),int(rng.integers(0,H))
    x=int(np.clip(cx-crop//2,0,W-crop)); y=int(np.clip(cy-crop//2,0,H-crop))
    xx=gray[y:y+crop,x:x+crop].copy();yy=lab[y:y+crop,x:x+crop].copy()
    if train_mode:
        if rng.random()<.5:xx=np.flip(xx,1);yy=np.flip(yy,1)
        if rng.random()<.5:xx=np.flip(xx,0);yy=np.flip(yy,0)
        k=int(rng.integers(0,4));xx=np.rot90(xx,k);yy=np.rot90(yy,k)
        xx=np.clip(xx.astype(np.float32)*rng.uniform(.82,1.18)+rng.uniform(-14,14)+
                   rng.normal(0,rng.uniform(0,3),xx.shape),0,255).astype(np.uint8)
        if rng.random()<.12:xx=cv2.GaussianBlur(xx,(3,3),rng.uniform(.2,1.1))
    tensor=make_channels(np.ascontiguousarray(xx)).astype(np.float32)/255.
    tensor=(tensor-.45)/.225
    return torch.from_numpy(np.ascontiguousarray(tensor.transpose(2,0,1))),torch.from_numpy(np.ascontiguousarray(yy[None].astype(np.float32)/255.))

class FilamentPatches(Dataset):
    def __init__(self,stems,length,train_mode=True,crop=512):
        self.stems=sorted(stems);self.length=int(length);self.train_mode=train_mode;self.crop=crop;self.epoch=0
    def set_epoch(self,e):self.epoch=int(e)
    def __len__(self):return self.length
    def __getitem__(self,idx):
        stem=self.stems[(idx*509+SEED+self.epoch*37)%len(self.stems)]
        return load_patches(stem,idx,self.train_mode,self.epoch,self.crop)

# Dry-run source fidelity check on a single authoritative training image.
if MODE not in ("SCORE_ONLY","INFER_ONLY"):
    d=FilamentPatches(SPLIT["train"],2)
    xx,yy=d[0]
    assert xx.shape==(3,SEMANTIC_CROP,SEMANTIC_CROP) and yy.shape==(1,SEMANTIC_CROP,SEMANTIC_CROP)
    assert np.isfinite(xx.numpy()).all() and (0<=yy.min()<=yy.max()<=1)
    print("Patch dry-run:",tuple(xx.shape),"foreground fraction:",float(yy.mean()))


In [ ]:
# 5 — Four distinct semantic model families; no old YOLO11n weights
class ConvRes(nn.Module):
    def __init__(self,a,b,stride=1):
        super().__init__()
        self.main=nn.Sequential(nn.Conv2d(a,b,3,stride,1,bias=False),nn.GroupNorm(min(8,b),b),nn.SiLU(),
                                nn.Conv2d(b,b,3,1,1,bias=False),nn.GroupNorm(min(8,b),b))
        self.skip=nn.Identity() if a==b and stride==1 else nn.Conv2d(a,b,1,stride,bias=False)
    def forward(self,x):return F.silu(self.main(x)+self.skip(x))

class ResUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.s0=nn.Sequential(ConvRes(3,base),ConvRes(base,base))
        self.s1=ConvRes(base,base*2,2)
        self.s2=ConvRes(base*2,base*4,2)
        self.s3=ConvRes(base*4,base*8,2)
        self.s4=ConvRes(base*8,base*12,2)
        self.d3=ConvRes(base*20,base*8)
        self.d2=ConvRes(base*12,base*4)
        self.d1=ConvRes(base*6,base*2)
        self.d0=ConvRes(base*3,base)
        self.head=nn.Conv2d(base,1,1)
    def forward(self,x):
        x0=self.s0(x);x1=self.s1(x0);x2=self.s2(x1);x3=self.s3(x2);x4=self.s4(x3)
        up=lambda x,y:F.interpolate(x,size=y.shape[-2:],mode="bilinear",align_corners=False)
        t=self.d3(torch.cat([up(x4,x3),x3],1))
        t=self.d2(torch.cat([up(t,x2),x2],1))
        t=self.d1(torch.cat([up(t,x1),x1],1))
        t=self.d0(torch.cat([up(t,x0),x0],1))
        return self.head(t)

class TimmFPN(nn.Module):
    def __init__(self,backbone,width=96,pretrained=False):
        super().__init__()
        self.pretrained_loaded=False
        if pretrained:
            try:
                self.encoder=timm.create_model(backbone,features_only=True,pretrained=True,in_chans=3)
                self.pretrained_loaded=True
                print("ImageNet pretrained encoder acquired:",backbone)
            except Exception as e:
                print("ImageNet pretrained encoder unavailable. Training FROM SCRATCH:",backbone,str(e)[:170])
                self.encoder=timm.create_model(backbone,features_only=True,pretrained=False,in_chans=3)
        else:self.encoder=timm.create_model(backbone,features_only=True,pretrained=False,in_chans=3)
        channels=self.encoder.feature_info.channels()
        # Most timm backbones expose 5 stages. Use 4 richest resolutions.
        channels=channels[-4:]
        self.lateral=nn.ModuleList([nn.Conv2d(c,width,1) for c in channels])
        self.smooth=nn.ModuleList([ConvRes(width,width) for _ in channels])
        self.head=nn.Sequential(ConvRes(width,width//2),nn.Conv2d(width//2,1,1))
    def forward(self,x):
        feats=self.encoder(x)[-4:]
        t=self.lateral[-1](feats[-1])
        for i in range(len(feats)-2,-1,-1):
            t=F.interpolate(t,size=feats[i].shape[-2:],mode="bilinear",align_corners=False)+self.lateral[i](feats[i])
            t=self.smooth[i](t)
        t=self.head(t)
        return F.interpolate(t,size=x.shape[-2:],mode="bilinear",align_corners=False)

def make_model(config,pretrained=False):
    if config["kind"]=="resunet":return ResUNet(32)
    return TimmFPN(config["backbone"],pretrained=pretrained)

print("Model families:",[r["name"] for r in SEMANTIC_RUNS])


In [ ]:
# 6 — Topology-aware loss and tracked checkpoint training; time budget is a CAP, not a promise
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

def soft_skeleton(x,iterations=3):
    sk=F.relu(x-F.max_pool2d(-x,3,1,1)*-1)
    cur=x
    for _ in range(iterations):
        erode=-F.max_pool2d(-cur,3,1,1)
        opened=F.max_pool2d(erode,3,1,1)
        cur=erode
        sk=torch.maximum(sk,F.relu(cur-opened))
    return sk

def filament_loss(logits,target):
    p=torch.sigmoid(logits)
    pos=torch.tensor(10.,device=logits.device)
    bce=F.binary_cross_entropy_with_logits(logits,target,pos_weight=pos)
    dims=(1,2,3)
    dice=1-(2*(p*target).sum(dims)+1)/((p+target).sum(dims)+1)
    sp=soft_skeleton(p);st=soft_skeleton(target)
    precision=(sp*target).sum(dims).add(1)/sp.sum(dims).add(1)
    sensitivity=(st*p).sum(dims).add(1)/st.sum(dims).add(1)
    cldice=1-(2*precision*sensitivity+1e-6)/(precision+sensitivity+1e-6)
    return .60*bce+.30*dice.mean()+.10*cldice.mean()

@torch.no_grad()
def val_patch_dice(model,loader):
    model.eval();vals=[]
    for x,y in loader:
        p=torch.sigmoid(model(x.to(DEVICE)))>.5
        y=y.to(DEVICE)>.5
        dice=(2*(p&y).sum((1,2,3))+1)/(p.sum((1,2,3))+y.sum((1,2,3))+1)
        vals.extend(dice.float().cpu().numpy())
    return float(np.mean(vals)) if vals else 0.

STAGE_RESULT={}

def run_one_semantic(cfg):
    global STAGE_RESULT
    name=cfg["name"]
    path=OUT/"models"/(name+"_best.pt")
    last=OUT/"models"/(name+"_last.pt")
    if not path.exists() and PREVIOUS_V16_ROOT:
        previous=Path(PREVIOUS_V16_ROOT)/"models"/(name+"_best.pt")
        if previous.exists():
            shutil.copy2(previous,path)
            print("Attached previous V16 trained checkpoint:",name)
    if path.exists():
        print("Existing trained checkpoint for",name,"reused (not retrained)")
        STAGE_RESULT[name]={"status":"reused","checkpoint":str(path)}
        return
    if not have_time(max(.7,cfg["stage_hours"]*.5)):
        STAGE_RESULT[name]={"status":"skipped_time"};return
    stage_start=time.monotonic()
    start_left=hours_left()
    batch=cfg.get("batch",SEMANTIC_BATCH)
    model=make_model(cfg,pretrained=TRY_IMAGENET_PRETRAINED).to(DEVICE)
    optimizer=AdamW(model.parameters(),lr=2.5e-4,weight_decay=0.01)
    schedule=CosineAnnealingLR(optimizer,T_max=max(1,cfg["epochs"]),eta_min=2e-5)
    scaler=torch.amp.GradScaler("cuda",enabled=(DEVICE.type=="cuda"))
    begin_epoch=0;best=-1.;initialization=bool(getattr(model,"pretrained_loaded",False))
    if last.exists():
        old=torch.load(last,map_location="cpu",weights_only=False)
        model.load_state_dict(old["model"]);optimizer.load_state_dict(old["optimizer"])
        schedule.load_state_dict(old["scheduler"])
        begin_epoch=int(old["epoch"])+1;best=float(old["best"])
        initialization=old.get("imagenet_pretrained",initialization)
        print("Resuming",name,"after epoch",begin_epoch)
    train_ds=FilamentPatches(SPLIT["train"],cfg["steps"]*batch,True,SEMANTIC_CROP)
    val_ds=FilamentPatches(SPLIT["tune"],64,False,SEMANTIC_CROP)
    valid=DataLoader(val_ds,batch_size=batch,num_workers=NUM_WORKERS,shuffle=False,pin_memory=True)
    log=[]
    try:
        for epoch in range(begin_epoch,int(cfg["epochs"])):
            if time.monotonic()-stage_start>cfg["stage_hours"]*3600 or not have_time(.18):break
            train_ds.set_epoch(epoch)
            train=DataLoader(train_ds,batch_size=batch,num_workers=NUM_WORKERS,shuffle=True,pin_memory=True)
            model.train();losses=[]
            for x,y in tqdm(train,desc=f"{name} epoch {epoch+1}",leave=False):
                if not have_time(.05):break
                x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda",enabled=DEVICE.type=="cuda"):
                    logits=model(x);loss=filament_loss(logits,y)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(),2.)
                scaler.step(optimizer);scaler.update()
                losses.append(float(loss.detach()))
            score=val_patch_dice(model,valid)
            schedule.step()
            state={"model":model.state_dict(),"epoch":epoch,"best":best,"config":cfg,
                   "imagenet_pretrained":initialization,"training_loss":np.mean(losses) if losses else None}
            if score>best:
                best=score
                state["best"]=best
                torch.save({"model":model.state_dict(),"config":cfg,"best_val_dice":best,
                            "imagenet_pretrained":initialization},path)
            state["best"]=best
            torch.save(state,last)
            log.append({"epoch":epoch+1,"train_loss":np.mean(losses) if losses else None,
                        "val_patch_dice":score,"hours_left":hours_left()})
            print(name,"epoch",epoch+1,"loss",round(float(np.mean(losses)),4) if losses else None,
                  "val_patch_dice",round(score,4),"best",round(best,4),"hours_left",round(hours_left(),2))
            if not losses:break
    except (RuntimeError,KeyboardInterrupt) as e:
        print("Training interrupted; last completed epoch was saved:",name,str(e)[:400])
        if isinstance(e,KeyboardInterrupt):raise
        if "out of memory" in str(e).lower():torch.cuda.empty_cache()
        else:raise
    finally:
        del model;gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
    pd.DataFrame(log).to_csv(OUT/"reports"/(name+"_training.csv"),index=False)
    STAGE_RESULT[name]={"status":"trained" if path.exists() else "not_completed","checkpoint":str(path),
                        "best_patch_dice":best,"imagenet_pretrained":initialization,"epochs_this_session":len(log)}
    save_json("semantic_stage_progress.json",STAGE_RESULT)

if MODE in ("FULL","SEMANTIC_ONLY") and RUN_SEMANTIC:
    for cfg in SEMANTIC_RUNS:
        try:run_one_semantic(cfg)
        except Exception as e:
            print("Architecture failed, continuing other models:",cfg["name"],repr(e))
            STAGE_RESULT[cfg["name"]]={"status":"failed","reason":repr(e)[:500]}
            save_json("semantic_stage_progress.json",STAGE_RESULT)
print("Semantic stage results:",STAGE_RESULT)


In [ ]:
# 7 — Independent instance learner: YOLO11s-seg / YOLO11m-seg (new checkpoints, not old 11n)
from ultralytics import YOLO

def yolo_polygon_from_rle(rr,epsilon=1.8,max_points=250):
    mask=coco.decode(rr).astype(np.uint8)
    contours,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    pieces=[]
    for cnt in contours:
        if cv2.contourArea(cnt)<30:continue
        simp=cv2.approxPolyDP(cnt,epsilon,True).reshape(-1,2)
        if len(simp)>max_points:
            ii=np.linspace(0,len(simp)-1,max_points).astype(int);simp=simp[ii]
        if len(simp)>=3:
            coords=np.clip(simp.astype(float)/[W,H],0,1)
            pieces.append((float(cv2.contourArea(cnt)),"0 "+" ".join(f"{a:.6f} {b:.6f}" for a,b in coords)))
    # YOLO segmentation labels cannot represent multiple disconnected islands as one polygon.
    return [max(pieces,key=lambda z:z[0])[1]] if pieces else []

def select_annotation_version(stem):
    versions=VERSION_BY_STEM[stem]
    counts=[len(a) for _,a in versions]
    midpoint=float(np.median(counts))
    return sorted(versions,key=lambda pair:(abs(len(pair[1])-midpoint),pair[0]))[0][1]

def make_yolo_dataset():
    root=OUT/"yolo_data"
    for split,stems in [("train",SPLIT["train"]),("val",SPLIT["tune"])]:
        images=root/"images"/split;labels=root/"labels"/split
        images.mkdir(parents=True,exist_ok=True);labels.mkdir(parents=True,exist_ok=True)
        for stem in tqdm(sorted(stems),desc="YOLO polygon dataset "+split):
            src=TRAIN[stem];dest=images/src.name;txt=labels/(stem+".txt")
            if not dest.exists():dest.symlink_to(src)
            if txt.exists():continue
            lines=[]
            for ann in select_annotation_version(stem):
                rr=annotation_rle(ann)
                if rr is not None:lines.extend(yolo_polygon_from_rle(rr))
            txt.write_text("\n".join(lines)+( "\n" if lines else ""))
    spec=root/"data.yaml"
    spec.write_text("path: "+str(root)+"\ntrain: images/train\nval: images/val\nnames:\n  0: filament\n")
    return spec

YOLO_STAGES={}
def run_yolo(name,hours):
    weights=OUT/"models"/(name+"_best.pt")
    if weights.exists():
        YOLO_STAGES[name]={"status":"reused","checkpoint":str(weights)};return
    if not have_time(max(.7,hours*.6)):
        YOLO_STAGES[name]={"status":"skipped_time"};return
    yaml_path=make_yolo_dataset()
    pretrained=False
    model_id=name.replace("_seg","-seg") # e.g. yolo11s-seg
    if TRY_OFFICIAL_YOLO_PRETRAINED:
        try:
            model=YOLO(model_id+".pt")
            pretrained=True
            print("Official pretrained YOLO loaded:",model_id)
        except Exception as e:
            print("Official YOLO weights unavailable; learning from scratch:",str(e)[:180])
            model=YOLO(model_id+".yaml")
    else:model=YOLO(model_id+".yaml")
    print("YOLO stage",name,"time budget hours",hours)
    try:
        model.train(data=str(yaml_path),epochs=115,time=float(hours),imgsz=1024,batch=4,
                    optimizer="AdamW",lr0=.002,lrf=.02,mosaic=.60,mixup=.05,
                    hsv_h=.005,hsv_s=.20,hsv_v=.20,fliplr=.5,flipud=.5,degrees=90,
                    workers=NUM_WORKERS,device=0,amp=True,patience=18,
                    project=str(OUT/"yolo_runs"),name=name,exist_ok=True,
                    cache=False,save=True,save_period=10,verbose=False)
    except (RuntimeError,KeyboardInterrupt) as e:
        print("YOLO interrupted. Looking for recoverable best/last weights:",repr(e)[:400])
        if isinstance(e,KeyboardInterrupt):raise
    candidates=[OUT/"yolo_runs"/name/"weights"/"best.pt",OUT/"yolo_runs"/name/"weights"/"last.pt"]
    found=next((p for p in candidates if p.is_file()),None)
    if found:
        shutil.copy2(found,weights)
        YOLO_STAGES[name]={"status":"trained","checkpoint":str(weights),"pretrained":pretrained}
    else:YOLO_STAGES[name]={"status":"not_completed","pretrained":pretrained}
    save_json("yolo_stage_progress.json",YOLO_STAGES)

if MODE in ("FULL","YOLO_ONLY") and RUN_YOLO:
    for name,hours in [("yolo11s_seg",YOLO_S_HOURS),("yolo11m_seg",YOLO_M_HOURS)]:
        if name=="yolo11m_seg" and not RUN_YOLO_M:continue
        try:run_yolo(name,hours)
        except Exception as e:
            print("YOLO stage failed, preserving earlier models:",name,repr(e))
            YOLO_STAGES[name]={"status":"failed","reason":repr(e)[:500]}
            save_json("yolo_stage_progress.json",YOLO_STAGES)
print("YOLO stages:",YOLO_STAGES)


In [ ]:
# 8 — Native-resolution semantic inference with overlapping 512px patches and Hann blending
from itertools import product
SEM_MODEL_CKPTS={r["name"]:OUT/"models"/(r["name"]+"_best.pt") for r in SEMANTIC_RUNS}
YOLO_MODEL_CKPTS={n:OUT/"models"/(n+"_best.pt") for n in ("yolo11s_seg","yolo11m_seg")}

def borrow_checkpoint_if_needed(p):
    if p.is_file():return p
    if PREVIOUS_V16_ROOT:
        prev=Path(PREVIOUS_V16_ROOT)/p.relative_to(OUT)
        if prev.is_file():return prev
    return None

def available_models():
    return {"semantic":[k for k,p in SEM_MODEL_CKPTS.items() if borrow_checkpoint_if_needed(p)],
            "yolo":[k for k,p in YOLO_MODEL_CKPTS.items() if borrow_checkpoint_if_needed(p)]}

LOADED={}
def get_sem_model(name):
    if name in LOADED:return LOADED[name]
    cfg=next(x for x in SEMANTIC_RUNS if x["name"]==name)
    checkpoint=borrow_checkpoint_if_needed(SEM_MODEL_CKPTS[name])
    assert checkpoint, name
    model=make_model(cfg,pretrained=False)
    payload=torch.load(checkpoint,map_location="cpu",weights_only=False)
    model.load_state_dict(payload["model"])
    model.eval().to(DEVICE)
    LOADED[name]=model
    return model

def release_models():
    LOADED.clear();gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

def starts(length=2048,window=512,step=416):
    ans=list(range(0,length-window+1,step))
    if ans[-1] != length-window:ans.append(length-window)
    return sorted(set(ans))

TILE_STARTS=starts(H,SEMANTIC_CROP,int(SEMANTIC_CROP*.80))
HANN=np.outer(np.hanning(SEMANTIC_CROP),np.hanning(SEMANTIC_CROP)).astype(np.float32)
HANN=np.maximum(.03,HANN)

@torch.inference_mode()
def predict_semantic_gray(model,gray,tta=USE_TTA,batch=4):
    assert gray.shape==(H,W)
    accum=np.zeros((H,W),np.float32);weight=np.zeros((H,W),np.float32)
    patches=[];locs=[]
    def flush():
        nonlocal patches,locs
        if not patches:return
        xx=torch.from_numpy(np.stack(patches)).to(DEVICE)
        with torch.amp.autocast("cuda",enabled=DEVICE.type=="cuda"):
            logits=model(xx)
            pp=torch.sigmoid(logits).float()
            if tta:
                pp=(pp+torch.flip(torch.sigmoid(model(torch.flip(xx,[-1]))).float(),[-1]))/2
        vals=pp[:,0].cpu().numpy()
        for z,(x,y) in zip(vals,locs):
            accum[y:y+SEMANTIC_CROP,x:x+SEMANTIC_CROP]+=z*HANN
            weight[y:y+SEMANTIC_CROP,x:x+SEMANTIC_CROP]+=HANN
        patches=[];locs=[]
    for y in TILE_STARTS:
        for x in TILE_STARTS:
            p=make_channels(gray[y:y+SEMANTIC_CROP,x:x+SEMANTIC_CROP]).astype(np.float32)/255.
            p=(p-.45)/.225
            patches.append(p.transpose(2,0,1).copy());locs.append((x,y))
            if len(patches)>=batch:flush()
    flush()
    return np.clip(accum/np.maximum(weight,1e-5),0,1)

def probability_file(model,stem,dataset):
    tag="val" if dataset=="val" else "test"
    p=OUT/"probabilities"/model/tag/(stem+".png")
    p.parent.mkdir(parents=True,exist_ok=True)
    if p.is_file():return p
    prev=None
    if PREVIOUS_V16_ROOT:
        c=Path(PREVIOUS_V16_ROOT)/"probabilities"/model/tag/(stem+".png")
        if c.exists():prev=c
    if prev:return prev
    gray=cv2.imread(str(TRAIN[stem] if dataset=="val" else TEST[stem]),cv2.IMREAD_GRAYSCALE)
    prob=predict_semantic_gray(get_sem_model(model),gray)
    cv2.imwrite(str(p),np.round(prob*65535).astype(np.uint16),[cv2.IMWRITE_PNG_COMPRESSION,3])
    return p

def read_prob(model,stem,dataset):
    f=probability_file(model,stem,dataset)
    arr=cv2.imread(str(f),cv2.IMREAD_UNCHANGED)
    assert arr is not None and arr.shape==(H,W)
    return arr.astype(np.float32)/(65535. if arr.dtype==np.uint16 else 255.)

print("Model checkpoints discovered:",available_models())


In [ ]:
# 9 — Instance reconstruction from semantic maps (morphology and fragmentation control)
# Coordinates, instance identities, and RLE format are generated anew; this is not the V7 recipe.

def semantic_instances(prob,threshold=.5,minimum_area=350,close_size=5,max_instances=140):
    foreground=(prob>=float(threshold)).astype(np.uint8)
    if close_size>=3:
        kernel=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(int(close_size),int(close_size)))
        foreground=cv2.morphologyEx(foreground,cv2.MORPH_CLOSE,kernel)
    n,labels,stats,_=cv2.connectedComponentsWithStats(foreground,8)
    eligible=[j for j in range(1,n) if int(stats[j,cv2.CC_STAT_AREA])>=minimum_area]
    eligible.sort(key=lambda j:int(stats[j,cv2.CC_STAT_AREA]),reverse=True)
    records=[]
    for j in eligible[:max_instances]:
        x,y,w,h,area=map(int,stats[j])
        # Avoid allocating a giant boolean per component when possible; only the crop is compared.
        tile=(labels[y:y+h,x:x+w]==j).astype(np.uint8)
        full=np.zeros((H,W),np.uint8)
        full[y:y+h,x:x+w]=tile
        rr=enc(full)
        records.append(rr)
    return records

def local_pq_counts(gt,pred):
    if not gt and not pred:return np.zeros(4,float)
    if not gt:return np.asarray([0,0,len(pred),0],float)
    if not pred:return np.asarray([0,0,0,len(gt)],float)
    mat=np.asarray(coco.iou(gt,pred,[0]*len(pred)),float)
    # Unique one-to-one matching with >.5 IoU (local approximation).
    rows,cols=linear_sum_assignment(-mat)
    keep=mat[rows,cols]>.5
    tp=int(keep.sum())
    return np.asarray([float(mat[rows[keep],cols[keep]].sum()),tp,len(pred)-tp,len(gt)-tp],float)

def counts_for_stem(stem,pred):
    return sum((local_pq_counts(gt,pred) for gt in GT[stem]),np.zeros(4,float))

def pq(counts):
    ss,tp,fp,fn=map(float,counts)
    denominator=tp+.5*(fp+fn)
    return float(ss/denominator) if denominator else 0.

def full_metrics(c):
    ss,tp,fp,fn=map(float,c)
    d=tp+.5*(fp+fn)
    return dict(PQ=pq(c),SQ=float(ss/tp) if tp else 0.,RQ=float(tp/d) if d else 0.,
                TP=int(tp),FP=int(fp),FN=int(fn))

# Self-test: a perfect known mask scores 1 and missed mask scores 0.
small=np.zeros((H,W),np.uint8);small[20:60,70:110]=1
r=enc(small)
assert pq(local_pq_counts([r],[r]))==1.
assert pq(local_pq_counts([r],[]))==0.
assert pq(local_pq_counts([],[r]))==0.
print("Strict one-to-one local PQ tests passed. Official evaluator parity NOT claimed.")


In [ ]:
# 10 — Semantic model shortlist: 24-image parameter screening, full 97-image scoring.
# Select on the tune split only. The previous audit was inspected and is NOT pristine.
SEARCH_GRID=[
    dict(threshold=t,minimum_area=a,close_size=c)
    for t in (.35,.45,.55,.65) for a in (200,400,650) for c in (1,5)
]
SCREEN_STEMS=sorted(SPLIT["tune"])[::4][:24]
TUNE_STEMS=sorted(SPLIT["tune"])[:VALIDATION_TUNE_LIMIT]
SEMANTIC_BOARD=[]
WINNER_SEM_PARAMS={}

def score_semantic_model(name):
    print("Semantic model PQ study:",name)
    release_models()  # only one network on GPU at a time
    screen=[]
    for s in tqdm(SCREEN_STEMS,desc=name+" 24-image screen"):
        if hours_left()<.08:break
        pm=read_prob(name,s,"val")
        for k,params in enumerate(SEARCH_GRID):
            pred=semantic_instances(pm,**params)
            vals=counts_for_stem(s,pred)
            screen.append((k,vals))
    if len(screen)<len(SEARCH_GRID):return None
    pooled={i:np.zeros(4,float) for i in range(len(SEARCH_GRID))}
    for i,val in screen:pooled[i]+=val
    top=sorted(pooled,key=lambda k:pq(pooled[k]),reverse=True)[:3]
    final={i:np.zeros(4,float) for i in top}
    processed=0
    for s in tqdm(TUNE_STEMS,desc=name+" full tune"):
        if hours_left()<.07:break
        pm=read_prob(name,s,"val")
        for i in top:
            final[i]+=counts_for_stem(s,semantic_instances(pm,**SEARCH_GRID[i]))
        processed+=1
    if processed!=len(TUNE_STEMS):
        print("Incomplete semantic tune scoring; cannot rank:",processed,"/",len(TUNE_STEMS))
        return None
    winner=max(top,key=lambda j:pq(final[j]))
    metric=full_metrics(final[winner])
    item={"model":name,"type":"semantic","params":SEARCH_GRID[winner],
          "images_scored":len(TUNE_STEMS),**metric}
    return item

if MODE in ("FULL","SEMANTIC_ONLY","SCORE_ONLY"):
    for name in available_models()["semantic"]:
        existing=OUT/"reports"/(name+"_pq.json")
        if existing.exists():item=json.loads(existing.read_text())
        else:
            try:item=score_semantic_model(name)
            except Exception as e:
                print("Skipping scoring for",name,repr(e)[:600]);item=None
            if item:save_json(name+"_pq.json",item)
        if item:
            SEMANTIC_BOARD.append(item)
            WINNER_SEM_PARAMS[name]=item["params"]
release_models()
if SEMANTIC_BOARD:
    board=pd.DataFrame(SEMANTIC_BOARD).sort_values("PQ",ascending=False)
    board.to_csv(OUT/"reports"/"semantic_leaderboard.csv",index=False)
    display(board[["model","PQ","SQ","RQ","TP","FP","FN","images_scored"]])
else:print("No trained semantic checkpoint available for evaluation in this run.")


In [ ]:
# 11 — Score NEW YOLO11s/m models by instance PQ, never against the old 0.34 submission.
YOLO_LOADED={}
def yolo_engine(name):
    if name not in YOLO_LOADED:
        checkpoint=borrow_checkpoint_if_needed(YOLO_MODEL_CKPTS[name])
        assert checkpoint
        YOLO_LOADED[name]=YOLO(str(checkpoint))
    return YOLO_LOADED[name]

def raw_yolo_predictions(name,stem,dataset="val"):
    folder=OUT/"probabilities"/name/dataset
    folder.mkdir(parents=True,exist_ok=True)
    cache=folder/(stem+".json")
    if cache.exists():return json.loads(cache.read_text())
    if PREVIOUS_V16_ROOT:
        other=Path(PREVIOUS_V16_ROOT)/"probabilities"/name/dataset/(stem+".json")
        if other.exists():return json.loads(other.read_text())
    image=TRAIN[stem] if dataset=="val" else TEST[stem]
    pred=yolo_engine(name).predict(str(image),imgsz=1024,conf=.10,iou=.7,
                                   retina_masks=True,device=0,max_det=140,verbose=False)[0]
    out=[]
    if pred.masks is not None and pred.boxes is not None:
        masks=pred.masks.data.detach().cpu().numpy()
        conf=pred.boxes.conf.detach().cpu().numpy()
        for p,score in zip(masks,conf):
            binary=(p>.5).astype(np.uint8)
            if binary.shape!=(H,W):binary=cv2.resize(binary,(W,H),interpolation=cv2.INTER_NEAREST)
            if int(binary.sum())<60:continue
            r=enc(binary)
            out.append({"counts":asc(r),"score":float(score),"area":int(coco.area(r))})
    tmp=cache.with_suffix(".tmp")
    tmp.write_text(json.dumps(out,separators=(",",":")))
    tmp.replace(cache)
    return out

def yolo_instances(raw,confidence=.30,minimum_area=350):
    selected=[];occupied=np.zeros((H,W),np.uint8)
    for a in sorted(raw,key=lambda d:-float(d["score"])):
        if float(a["score"])<confidence or int(a["area"])<minimum_area:continue
        r=as_rle(a["counts"])
        if selected:
            ious=np.asarray(coco.iou([r],selected,[0]*len(selected)))[0]
            if len(ious) and ious.max()>.65:continue
        mask=coco.decode(r).astype(np.uint8)
        free=mask&(occupied==0)
        if int(free.sum())<minimum_area:continue
        rr=enc(free)
        selected.append(rr);occupied|=free
    return selected

YOLO_GRID=[dict(confidence=t,minimum_area=a) for t in (.18,.30,.45,.60) for a in (200,400,700)]
YOLO_BOARD=[]
def score_yolo(name):
    # 24-image screen followed by full 97-image tune. Different architectures, same protocol.
    ranking={i:np.zeros(4,float) for i in range(len(YOLO_GRID))}
    for s in tqdm(SCREEN_STEMS,desc=name+" YOLO screen"):
        if hours_left()<.08:return None
        raw=raw_yolo_predictions(name,s)
        for i,param in enumerate(YOLO_GRID):
            ranking[i]+=counts_for_stem(s,yolo_instances(raw,**param))
    top=sorted(ranking,key=lambda i:pq(ranking[i]),reverse=True)[:3]
    final={i:np.zeros(4,float) for i in top}
    processed=0
    for s in tqdm(TUNE_STEMS,desc=name+" YOLO full tune"):
        if hours_left()<.07:break
        raw=raw_yolo_predictions(name,s)
        for i in top:final[i]+=counts_for_stem(s,yolo_instances(raw,**YOLO_GRID[i]))
        processed+=1
    if processed!=len(TUNE_STEMS):
        print("Incomplete YOLO tune scoring; not eligible for ranking:",processed,"of",len(TUNE_STEMS))
        return None
    winner=max(top,key=lambda i:pq(final[i]))
    return {"model":name,"type":"yolo","params":YOLO_GRID[winner],
            "images_scored":processed,**full_metrics(final[winner])}

if MODE in ("FULL","YOLO_ONLY","SCORE_ONLY"):
    for name in available_models()["yolo"]:
        saved=OUT/"reports"/(name+"_pq.json")
        if saved.exists():result=json.loads(saved.read_text())
        else:
            try:result=score_yolo(name)
            except Exception as e:
                print("Failed YOLO evaluation for",name,repr(e)[:300]);result=None
            if result:save_json(name+"_pq.json",result)
        if result:YOLO_BOARD.append(result)
if YOLO_BOARD:
    df=pd.DataFrame(YOLO_BOARD).sort_values("PQ",ascending=False)
    df.to_csv(OUT/"reports"/"yolo_leaderboard.csv",index=False)
    display(df[["model","PQ","SQ","RQ","TP","FP","FN","images_scored"]])
print("Independent model evaluation complete:",len(SEMANTIC_BOARD),"semantic",len(YOLO_BOARD),"instance")


In [ ]:
# 12 — Hybrid architectures: semantic ensembled probability / YOLO→semantic mask refinement + rescue
# This is a NEW proposal/refinement model family, not a recovered historical instance postprocessor.

SEMANTIC_BOARD=SEMANTIC_BOARD if "SEMANTIC_BOARD" in globals() else []
YOLO_BOARD=YOLO_BOARD if "YOLO_BOARD" in globals() else []

if MODE=="INFER_ONLY":
    prior=(Path(PREVIOUS_V16_ROOT)/"reports"/"new_model_selection.json") if PREVIOUS_V16_ROOT else (OUT/"reports"/"new_model_selection.json")
    if prior.exists():
        past=json.loads(prior.read_text()).get("ranked_models",[])
        if not SEMANTIC_BOARD: SEMANTIC_BOARD=[r for r in past if r["type"]=="semantic"]
        if not YOLO_BOARD: YOLO_BOARD=[r for r in past if r["type"]=="yolo"]
VALID_SEM=sorted(SEMANTIC_BOARD,key=lambda x:-x["PQ"])
VALID_YOLO=sorted(YOLO_BOARD,key=lambda x:-x["PQ"])
HYBRID_BOARD=[]

def ensemble_prob(stem,dataset="val",models=None):
    models=models or [x["model"] for x in VALID_SEM[:2]]
    if not models:raise ValueError("No semantic models for ensemble")
    return np.mean([read_prob(name,stem,dataset) for name in models],axis=0)

def semantic_refine_yolo(yolo_rles,prob,sem_thresh=.50,expand=7,rescue_area=500):
    refined=[]
    all_occ=np.zeros((H,W),np.uint8)
    disk=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(expand,expand))
    for rr in yolo_rles:
        original=coco.decode(rr).astype(np.uint8)
        expanded=cv2.dilate(original,disk)
        candidate=(original | (expanded & (prob>=sem_thresh).astype(np.uint8)))
        candidate&=(all_occ==0).astype(np.uint8)
        if int(candidate.sum())<200:continue
        refined.append(enc(candidate));all_occ|=candidate
    # Rescue only separated connected components; avoid arbitrarily splitting/merging YOLO masks.
    extras=semantic_instances(prob,threshold=max(.50,sem_thresh),minimum_area=rescue_area,close_size=5)
    for rr in extras:
        mask=coco.decode(rr).astype(np.uint8)
        if (mask & all_occ).sum()/max(1,mask.sum())>.15:continue
        free=mask & (all_occ==0)
        if int(free.sum())<rescue_area:continue
        refined.append(enc(free));all_occ|=free
    return refined

HYBRID_CANDIDATES=[]
if len(VALID_SEM)>=2:
    HYBRID_CANDIDATES += [dict(name="ensemble_top2",kind="ensemble",threshold=t,minimum_area=a,close_size=5)
                          for t in (.40,.50,.60) for a in (300,600)]
if VALID_YOLO and VALID_SEM:
    HYBRID_CANDIDATES += [dict(name="yolo_sem_refine",kind="refine",threshold=t,minimum_area=a,close_size=5)
                          for t in (.45,.55,.65) for a in (350,650)]

BEST_HYBRID=None
if HYBRID_CANDIDATES and MODE in ("FULL","SCORE_ONLY","INFER_ONLY") and hours_left()>.40:
    screen={i:np.zeros(4,float) for i in range(len(HYBRID_CANDIDATES))}
    for s in tqdm(SCREEN_STEMS,desc="Ensemble / hybrid screen"):
        pp=ensemble_prob(s,"val") if len(VALID_SEM)>=2 else read_prob(VALID_SEM[0]["model"],s,"val")
        base=None
        if VALID_YOLO:
            yy=VALID_YOLO[0]
            base=yolo_instances(raw_yolo_predictions(yy["model"],s),**yy["params"])
        for i,param in enumerate(HYBRID_CANDIDATES):
            if param["kind"]=="ensemble":
                rr=semantic_instances(pp,param["threshold"],param["minimum_area"],5)
            else:rr=semantic_refine_yolo(base,pp,sem_thresh=param["threshold"],
                                          rescue_area=param["minimum_area"])
            screen[i]+=counts_for_stem(s,rr)
    top=sorted(screen,key=lambda i:pq(screen[i]),reverse=True)[:2]
    totals={i:np.zeros(4,float) for i in top}
    done=0
    for s in tqdm(TUNE_STEMS,desc="Hybrid full tune"):
        if hours_left()<.08:break
        pp=ensemble_prob(s,"val") if len(VALID_SEM)>=2 else read_prob(VALID_SEM[0]["model"],s,"val")
        base=None
        if VALID_YOLO:
            yy=VALID_YOLO[0]
            base=yolo_instances(raw_yolo_predictions(yy["model"],s),**yy["params"])
        for i in top:
            param=HYBRID_CANDIDATES[i]
            rr=semantic_instances(pp,param["threshold"],param["minimum_area"],5) if param["kind"]=="ensemble" else semantic_refine_yolo(base,pp,param["threshold"],rescue_area=param["minimum_area"])
            totals[i]+=counts_for_stem(s,rr)
        done+=1
    if done==len(TUNE_STEMS):
        winner=max(top,key=lambda i:pq(totals[i]))
        BEST_HYBRID={"model":HYBRID_CANDIDATES[winner]["name"],"type":"hybrid",
                     "params":HYBRID_CANDIDATES[winner],"images_scored":done,
                     **full_metrics(totals[winner])}
        HYBRID_BOARD.append(BEST_HYBRID)
        save_json("hybrid_pq.json",BEST_HYBRID)
        print("Best hybrid:",BEST_HYBRID)
    else:print("Incomplete hybrid evaluation, cannot select.")
release_models()


In [ ]:
# 13 — Select the BEST COMPLETED NEW model, inspect per-group validation, optional old audit diagnostic
ALL_MODELS=sorted(SEMANTIC_BOARD+YOLO_BOARD+HYBRID_BOARD,key=lambda x:-x["PQ"])
selection_path=OUT/"reports"/"new_model_selection.json"
if not ALL_MODELS and selection_path.exists():
    previous=json.loads(selection_path.read_text())
    ALL_MODELS=previous.get("ranked_models",[])
if not ALL_MODELS:
    print("No fully validated new model yet; DO NOT submit an unvalidated CSV. Resume saved checkpoints.")
else:
    leaderboard=pd.DataFrame([{k:v for k,v in m.items() if k!="params"} for m in ALL_MODELS])
    display(leaderboard[["model","type","PQ","SQ","RQ","TP","FP","FN","images_scored"]])
    winner=ALL_MODELS[0]
    print("BEST NEW ARCHITECTURE:",winner["model"],"PQ",winner["PQ"],"params",winner["params"])
    plan={"selected_model":winner,"ranked_models":ALL_MODELS,
          "metric":"LOCAL one-to-one IoU>.5 PQ; official evaluator parity UNVERIFIED",
          "validation_split":"competition train: 513 training / 97 model selection / 97 old audit diagnostic",
          "caution":"Validation PQ is not expected Kaggle public PQ; never select by test labels",
          "historical_034_used_to_select":False,
          "test_label_access":False}
    save_json("new_model_selection.json",plan)


In [ ]:
# 13b — Scientific error analysis of selected new architecture on the 97 tune images.
# This diagnostic can be resumed later; no test ground truth is ever accessed.
if ALL_MODELS and hours_left()>.25:
    winner=ALL_MODELS[0]
    diagnostics=[]
    for s in tqdm(TUNE_STEMS,desc="New model tune failure analysis"):
        if hours_left()<.16:break
        if winner["type"]=="semantic":
            pr=semantic_instances(read_prob(winner["model"],s,"val"),**winner["params"])
        elif winner["type"]=="yolo":
            pr=yolo_instances(raw_yolo_predictions(winner["model"],s,"val"),**winner["params"])
        else:
            pa=winner["params"]
            pp=ensemble_prob(s,"val") if len(VALID_SEM)>=2 else read_prob(VALID_SEM[0]["model"],s,"val")
            if pa["kind"]=="ensemble":pr=semantic_instances(pp,pa["threshold"],pa["minimum_area"],pa["close_size"])
            else:
                primary=VALID_YOLO[0]
                originals=yolo_instances(raw_yolo_predictions(primary["model"],s,"val"),**primary["params"])
                pr=semantic_refine_yolo(originals,pp,pa["threshold"],rescue_area=pa["minimum_area"])
        cnt=counts_for_stem(s,pr)
        diagnostics.append(dict(stem=s,acquisition_day=s[:8],instances=len(pr),**full_metrics(cnt)))
    if diagnostics:
        df=pd.DataFrame(diagnostics).sort_values("PQ")
        df.to_csv(OUT/"reports"/"selected_model_per_image_pq.csv",index=False)
        print("Worst 10 tune images (diagnostic):")
        display(df.head(10)[["stem","PQ","TP","FP","FN","instances"]])
        print("Best 10 tune images:")
        display(df.tail(10)[["stem","PQ","TP","FP","FN","instances"]])


In [ ]:
# 14 — New-model test submission; no 0.34 fallback and no use of hidden labels.
# This cell is safe to rerun; once checkpoints are attached it can run INFER_ONLY.

selection_path=OUT/"reports"/"new_model_selection.json"
if not selection_path.exists() and PREVIOUS_V16_ROOT:
    backup=Path(PREVIOUS_V16_ROOT)/"reports"/"new_model_selection.json"
    if backup.exists():selection_path=backup

selected=None
if selection_path.exists():selected=json.loads(selection_path.read_text())["selected_model"]


def predict_selected(stem,selection):
    name=selection["model"]
    if selection["type"]=="semantic":
        prob=read_prob(name,stem,"test")
        return semantic_instances(prob,**selection["params"])
    if selection["type"]=="yolo":
        return yolo_instances(raw_yolo_predictions(name,stem,"test"),**selection["params"])
    params=selection["params"]
    pp=ensemble_prob(stem,"test") if len(VALID_SEM)>=2 else read_prob(VALID_SEM[0]["model"],stem,"test")
    if params["kind"]=="ensemble":
        return semantic_instances(pp,params["threshold"],params["minimum_area"],params["close_size"])
    primary=VALID_YOLO[0]
    base=yolo_instances(raw_yolo_predictions(primary["model"],stem,"test"),**primary["params"])
    return semantic_refine_yolo(base,pp,params["threshold"],rescue_area=params["minimum_area"])

if selected and MODE in ("FULL","INFER_ONLY","SCORE_ONLY","YOLO_ONLY","SEMANTIC_ONLY"):
    if hours_left()<.02:print("Run time exhausted; run INFER_ONLY in a new session with saved model outputs attached.")
    else:
        rows=[];per=[]
        for stem in tqdm(sorted(TEST),desc="NEW MODEL held-out test prediction"):
            rr=predict_selected(stem,selected)
            per.append(dict(stem=stem,instances=len(rr)))
            for i,r in enumerate(rr,1):
                if int(coco.area(r))<1:continue
                rows.append((f"{stem}_{i}",asc(r)))
        outdf=pd.DataFrame(rows,columns=["filament_id","segmentation_rle"])
        assert len(outdf)>=200, "Fewer than 200 predicted filaments across 180 images: inspect model; do not submit empty/degenerate result"
        assert outdf["filament_id"].is_unique
        assert outdf["segmentation_rle"].astype(bool).all()
        assert set(outdf["filament_id"].str.rsplit("_",n=1).str[0]).issubset(set(TEST))
        assert len(per)==len(TEST)
        dest=OUT/"submissions"/"V16_NEW_MODEL_submission.csv"
        outdf.to_csv(dest,index=False)
        # The canonical file is the NEW architecture output, not an old-file rollback.
        shutil.copy2(dest,"/kaggle/working/submission.csv")
        pd.DataFrame(per).to_csv(OUT/"reports"/"test_instance_counts.csv",index=False)
        save_json("new_submission_report.json",{"chosen_model":selected,"rows":len(outdf),
                 "images_with_predictions":sum(x["instances"]>0 for x in per),
                 "sha256":file_hash(dest),"path":str(dest),
                 "old_034_submission_reused":False,
                 "test_ground_truth_used":False})
        print("NEW-MODEL CSV READY:",dest,"rows",len(outdf))
        print("Upload only after inspecting model diagnostics and competition submission constraints.")
else:print("No new-model selection is available. Resume scoring before submitting.")
release_models()


In [ ]:
# 15 — Inventory and reproducibility pack; careful Kaggle restart workflow
import zipfile
report_files=sorted((OUT/"reports").glob("*"))
print("Reports:")
for p in report_files:
    if p.is_file():print(f"  {p.name:40s} {p.stat().st_size/1024:.1f} KiB")
print("Saved models:")
for p in sorted((OUT/"models").glob("*.pt")):
    print("  ",p.name,round(p.stat().st_size/1024**2,1),"MiB")
print("Available new-model submissions:",list((OUT/"submissions").glob("*.csv")))

# Lightweight handoff ZIP: reports, validation ledgers, and new submission CSV.
# Full model weights should be saved as a Kaggle dataset (too large for small ZIP).
zip_path=Path("/kaggle/working/SOLAR_V16_REPORTS_AND_SUBMISSION.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED,compresslevel=5) as z:
    for parent in [OUT/"reports",OUT/"submissions"]:
        for file in parent.rglob("*"):
            if file.is_file():z.write(file,arcname=str(file.relative_to(OUT)))
print("Handoff:",zip_path)
print("Remaining Kaggle time by this notebook's monotonic clock (hours):",round(hours_left(),2))


## Session strategy and interpretation

**What you are training:** ResUNet-32 with a topology-aware loss, FPN with ImageNet-pretrained ResNet34/EfficientNetB0/ConvNeXt-Tiny backbones when accessible (otherwise explicitly train from scratch), and two independent YOLO11s/m instance segmentation learners; then semantic ensembling and YOLO-guided semantic refinement. Every checkpoint and validation table is a new result, not a re-scored V7 prediction.

**GPU limits:** T4×2 does not automatically double notebook speed. The training code deliberately uses GPU 0 and mixed precision, keeping single-T4 compatibility. If time runs out, the next session can load model weights by attaching the entire previous V16 output dataset, setting `PREVIOUS_V16_ROOT`, and using `MODE='SCORE_ONLY'` or `MODE='INFER_ONLY'`. If you want to continue partially trained epochs, copy the previous `models/*_last.pt` into a writable V16 output first. **Do not pretend this is an across-session live continuation unless checkpoints are actually attached.**

**Output hierarchy:** `solar_v16_new_models/models/` trained weights; `reports/semantic_leaderboard.csv`, `yolo_leaderboard.csv`, `new_model_selection.json`, `new_submission_report.json`; `submissions/V16_NEW_MODEL_submission.csv`. The final CSV is generated from only the competition's H-alpha test images. No external competition-test masks may be consulted.

**Scientific validity:** Validation PQ is a proxy until matched against the official self-evaluation notebook. Do not interpret a new score above 0.5 as guaranteed LB PQ, or a 0.34 baseline as a ceiling. Report TP, FP, FN, SQ, RQ, instance counts, failure cases, and morphology. The old audit subset was previously inspected and should not be advertised as independent confirmation.